# LCT FINAL80 — RF-DETR Medium @640

Финальный deployment-training run.

Берём **external RF-DETR Medium @640 checkpoint ДО target fine-tuning** и обучаем на всех доступных organizer development labels:

- 65 former TRAIN
- 15 former VAL
- итого 80 изображений

20 TEST изображений не используются.

Бывший VAL теперь входит в train, поэтому у FINAL80 нет честного held-out validation score.
RF-DETR технически требует valid split, поэтому notebook создаёт `valid` как зеркало тех же 80 изображений.
Этот in-sample metric нельзя публиковать как generalization quality.

Фиксированный recipe:
- resolution 640
- 30 epochs
- lr 2e-5
- lr_encoder 1e-5
- EMA on
- early stopping off
- eval_interval 30
- seed 42

Deployment weights: **`last_ema.pth` после 30-й эпохи**.

## Kaggle Inputs

1. `grigoriyyy/lct-organizer100-split-v1`
2. Notebook Output от `lct_rfdetr_medium_640_finetune.ipynb` — external Medium@640, external mAP≈0.6984.

НЕ использовать как старт:
- `lct_target_ft_rfdetr_medium640`
- continuation
- FINAL80
- crop/organizer outputs


In [1]:
!nvidia-smi
!pip install -q "rfdetr[train]==1.10.1" pyyaml pandas pillow

from pathlib import Path
import json, time, random, shutil, hashlib
import numpy as np
import pandas as pd
import torch
import yaml
from PIL import Image
import rfdetr
from rfdetr import RFDETRMedium

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

CLASS_NAMES=[
    "excavator","dump_truck","truck","loader","bulldozer","motor_grader",
    "roller","concrete_mixer","telehandler","piling_machine",
    "crane_manipulator","mobile_crane","tower_crane"
]

INPUT=Path("/kaggle/input")
WORK=Path("/kaggle/working")
RF_DATA=WORK/"organizer_final80_rfdetr"
TRAIN_OUTPUT=WORK/"final80_rfdetr_medium640"
DEPLOY_DIR=WORK/"final80_deployment"

for p in [RF_DATA,TRAIN_OUTPUT,DEPLOY_DIR]:
    if p.exists(): shutil.rmtree(p)
DEPLOY_DIR.mkdir(parents=True,exist_ok=True)

MANUAL_EXTERNAL_CKPT=None

print("rfdetr:",getattr(rfdetr,"__version__","unknown"))
print("torch:",torch.__version__)
print("cuda:",torch.cuda.is_available())
if torch.cuda.is_available(): print("gpu:",torch.cuda.get_device_name(0))


Mon Sep 28 21:46:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
def normalize_names(raw):
    if isinstance(raw,dict):
        return [raw[i] if i in raw else raw[str(i)] for i in range(len(raw))]
    return list(raw)

dataset_candidates=[]
for y in INPUT.rglob("data.yaml"):
    root=y.parent
    if ((root/"train"/"images").exists() and (root/"train"/"labels").exists()
        and (root/"val"/"images").exists() and (root/"val"/"labels").exists()
        and (root/"test"/"images").exists()):
        dataset_candidates.append(y)

preferred=[x for x in dataset_candidates if "lct-organizer100-split-v1" in x.as_posix().lower() or "organizer100_split" in x.as_posix().lower()]
assert preferred or dataset_candidates, "Attach grigoriyyy/lct-organizer100-split-v1"

DATA_YAML=(preferred or dataset_candidates)[0]
DATA_ROOT=DATA_YAML.parent

with open(DATA_YAML,"r",encoding="utf-8") as f:
    cfg=yaml.safe_load(f)

assert normalize_names(cfg["names"])==CLASS_NAMES, "Taxonomy mismatch"

IMAGE_EXTS={".jpg",".jpeg",".png",".webp",".bmp",".tif",".tiff"}
def list_images(split):
    p=DATA_ROOT/split/"images"
    return sorted(x for x in p.iterdir() if x.is_file() and x.suffix.lower() in IMAGE_EXTS)

train_images=list_images("train")
val_images=list_images("val")
test_count=sum(1 for x in (DATA_ROOT/"test"/"images").iterdir() if x.is_file() and x.suffix.lower() in IMAGE_EXTS)

assert len(train_images)==65
assert len(val_images)==15
assert test_count==20
assert {x.name for x in train_images}.isdisjoint({x.name for x in val_images})

dev_images=[("train",p) for p in train_images]+[("val",p) for p in val_images]
assert len(dev_images)==80

print("DATA_ROOT:",DATA_ROOT)
print("former TRAIN:",len(train_images))
print("former VAL:",len(val_images))
print("FINAL80 training pool:",len(dev_images))
print("TEST locked:",test_count)


DATA_ROOT: /kaggle/input/datasets/grigoriyyy/lct-organizer100-split-v1/organizer100_split
former TRAIN: 65
former VAL: 15
FINAL80 training pool: 80
TEST locked: 20


In [3]:
if MANUAL_EXTERNAL_CKPT:
    EXTERNAL_CKPT=Path(MANUAL_EXTERNAL_CKPT)
    assert EXTERNAL_CKPT.exists()
else:
    reject=["target_ft","target-ft","organizer100","continue","continuation","final80","branch_aware","crop_val"]
    all_medium640=[]
    EXTERNAL_CKPT=None

    for fn in ["checkpoint_best_ema.pth","checkpoint_best_total.pth","checkpoint_best_regular.pth"]:
        hits=[]
        for p in INPUT.rglob(fn):
            low=p.as_posix().lower()
            if "medium" not in low or "640" not in low: continue
            all_medium640.append(p)
            if any(t in low for t in reject): continue
            hits.append(p)
        if hits:
            def rank(p):
                low=p.as_posix().lower()
                score=0
                if "rfdetr" in low: score-=10
                if "medium" in low: score-=10
                if "640" in low: score-=10
                if "finetune" in low or "fine-tune" in low: score-=5
                return (score,len(p.as_posix()))
            EXTERNAL_CKPT=sorted(hits,key=rank)[0]
            break

    print("Medium640 candidates:")
    for p in sorted(set(all_medium640),key=lambda x:x.as_posix()):
        print(" ",p)

assert EXTERNAL_CKPT is not None, "Attach external Medium640 Notebook Output or set MANUAL_EXTERNAL_CKPT"

low=EXTERNAL_CKPT.as_posix().lower()
forbidden=["target_ft","target-ft","continue","continuation","final80","crop_val"]
assert not any(t in low for t in forbidden), f"Wrong start checkpoint: {EXTERNAL_CKPT}"

print("\nUSING EXTERNAL CHECKPOINT:")
print(EXTERNAL_CKPT)


Medium640 candidates:
  /kaggle/input/notebooks/maryewwe/lct-rfdetr-medium-640-finetune/rfdetr_medium_640_finetune/checkpoint_best_ema.pth

USING EXTERNAL CHECKPOINT:
/kaggle/input/notebooks/maryewwe/lct-rfdetr-medium-640-finetune/rfdetr_medium_640_finetune/checkpoint_best_ema.pth


In [4]:
for split in ("train","valid"):
    (RF_DATA/split/"images").mkdir(parents=True,exist_ok=True)
    (RF_DATA/split/"labels").mkdir(parents=True,exist_ok=True)

manifest=[]
for source_split,img in dev_images:
    lab=DATA_ROOT/source_split/"labels"/f"{img.stem}.txt"
    assert lab.exists(),lab
    for dst in ("train","valid"):
        shutil.copy2(img,RF_DATA/dst/"images"/img.name)
        shutil.copy2(lab,RF_DATA/dst/"labels"/lab.name)
    manifest.append({"source_split":source_split,"image_name":img.name,"label_name":lab.name})

pd.DataFrame(manifest).to_csv(DEPLOY_DIR/"final80_manifest.csv",index=False)

rf_yaml={
    "path":".",
    "train":"train/images",
    "val":"valid/images",
    "names":{i:n for i,n in enumerate(CLASS_NAMES)},
    "nc":13,
}
(RF_DATA/"data.yaml").write_text(yaml.safe_dump(rf_yaml,sort_keys=False,allow_unicode=True),encoding="utf-8")

assert len(list((RF_DATA/"train"/"images").iterdir()))==80
assert len(list((RF_DATA/"valid"/"images").iterdir()))==80
assert not (RF_DATA/"test").exists()

print("train=80")
print("valid mirror=80")
print("IMPORTANT: valid mirror is IN-SAMPLE and must not be reported as generalization.")


train=80
valid mirror=80
IMPORTANT: valid mirror is IN-SAMPLE and must not be reported as generalization.


In [5]:
model=RFDETRMedium(
    pretrain_weights=str(EXTERNAL_CKPT),
    resolution=640,
    num_classes=13,
)

t0=time.perf_counter()

model.train(
    dataset_dir=str(RF_DATA),
    dataset_file="yolo",
    epochs=30,
    resolution=640,
    batch_size="auto",
    grad_accum_steps=1,
    num_workers=4,
    output_dir=str(TRAIN_OUTPUT),
    seed=SEED,

    lr=2e-5,
    lr_encoder=1e-5,

    use_ema=True,
    eval_base_model=False,

    checkpoint_interval=5,
    early_stopping=False,
    eval_interval=30,

    skip_best_epochs=0,
    run_test=False,
    tensorboard=True,
    wandb=False,
    progress_bar="tqdm",
)

print("Training hours:",(time.perf_counter()-t0)/3600)


[2026-09-28 21:48:13] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-09-28 21:48:13] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-09-28 21:48:19] [WARNING] rf-detr - [auto-batch] 1/509 trainable params received no gradient from the synthetic probe batch; their optimizer state is not modeled, so the returned batch size may be too permissive.
/usr/local/lib/python3.12/dist-packages/rfdetr/models/_assignment.py:76: RuntimeWarning: Triton linear-assignment support is unavailable for this CUDA input; using SciPy on CPU.
  assignment = torch_linear_assignment.batch_linear_assignment(stacked)


[2026-09-28 21:48:30] [INFO] rf-detr - [auto-batch] Applied EMA headroom (0.70): safe_micro_batch=3
[2026-09-28 21:48:30] [INFO] rf-detr - [auto-batch] device=Tesla T4 world_size=1 segmentation=False probe_resolution=800 max_targets_per_image=100
[2026-09-28 21:48:30] [INFO] rf-detr - [auto-batch] safe_micro_batch=3 grad_accum_steps=6 effective_batch_per_device=18 global_effective_batch=18
[2026-09-28 21:48:30] [INFO] rf-detr - [auto-batch] This probe estimates train-step-safe micro-batch only.
[2026-09-28 21:48:30] [INFO] rf-detr - [auto-batch] Validation/test (especially segmentation mask eval) may require more memory.
[2026-09-28 21:48:30] [INFO] rf-detr - [auto-batch] resolved train config: batch_size=3 grad_accum_steps=6 effective_batch_size=18


[2026-09-28 21:48:30] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-09-28 21:48:30] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


[2026-09-28 21:48:48] [INFO] rf-detr - Using multi-scale training with square resize and scales: [800]
[2026-09-28 21:48:48] [INFO] rf-detr - Built 1 Albumentations transforms from config
[2026-09-28 21:48:48] [INFO] rf-detr - Built 1 Albumentations transforms from config
creating index...
index created!
[2026-09-28 21:48:48] [INFO] rf-detr - Using multi-scale training with square resize and scales: [800]
creating index...
index created!


/usr/local/lib/python3.12/dist-packages/pytorch_lightning/callbacks/model_checkpoint.py:881: Checkpoint directory /kaggle/working/final80_rfdetr_medium640 exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loading `train_dataloader` to estimate number of stepping batches.


...

[2026-09-28 22:02:56] [INFO] rf-detr - Skipping validation-loss computation: compute_val_loss='auto' found no scheduler or callback monitoring 'val/loss', so no 'val/loss' metric is logged this run. Set compute_val_loss=True to log it every validation epoch (for example to read the curve from metrics.csv, TensorBoard, or Weights & Biases).


Output()

[2026-09-28 22:03:03] [INFO] rf-detr - Best EMA metric improved to 0.3831 (epoch 29)


`Trainer.fit` stopped: `max_epochs=30` reached.


[2026-09-28 22:03:10] [INFO] rf-detr - Best total checkpoint saved from EMA (regular=0.0000, ema=0.3831)
Training hours: 0.24888641423694444


In [6]:
print("Training output files:")
for p in sorted(TRAIN_OUTPUT.rglob("*")):
    if p.is_file(): print(" ",p.relative_to(TRAIN_OUTPUT))

FINAL_EMA=TRAIN_OUTPUT/"last_ema.pth"
assert FINAL_EMA.exists(), (
    "Expected last_ema.pth. Do NOT silently substitute checkpoint_best_*; inspect output/version."
)

FINAL_MODEL=DEPLOY_DIR/"final_model.pth"
shutil.copy2(FINAL_EMA,FINAL_MODEL)

for name in ["metrics.csv","training_config.json","checkpoint_best_total.pth","checkpoint_best_ema.pth"]:
    src=TRAIN_OUTPUT/name
    if src.exists():
        dst=("IN_SAMPLE_"+name) if name.startswith("checkpoint_best") else name
        shutil.copy2(src,DEPLOY_DIR/dst)

print("FINAL MODEL:",FINAL_MODEL)


Training output files:
  checkpoint_29.ckpt
  checkpoint_best_ema.pth
  checkpoint_best_total.pth
  events.out.tfevents.1790632128.5a9be755300f.23.0
  hparams.yaml
  last.ckpt
  last_ema.pth
  metrics.csv
  training_config.json
FINAL MODEL: /kaggle/working/final80_deployment/final_model.pth


In [7]:
def sha256_file(path,chunk=1024*1024):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        while True:
            b=f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()

meta={
    "model":"RF-DETR Medium",
    "resolution":640,
    "num_classes":13,
    "class_names":CLASS_NAMES,
    "seed":42,
    "epochs":30,
    "lr":2e-5,
    "lr_encoder":1e-5,
    "use_ema":True,
    "deployment_weights":"last_ema.pth copied as final_model.pth",
    "external_checkpoint":str(EXTERNAL_CKPT),
    "external_checkpoint_sha256":sha256_file(EXTERNAL_CKPT),
    "final_model_sha256":sha256_file(FINAL_MODEL),
    "development_images":80,
    "former_train_images":65,
    "former_val_images":15,
    "test_images_used":0,
    "validation_note":"No held-out validation exists for FINAL80; valid is an 80-image train mirror and is infrastructure-only.",
    "historical_reference_only":{
        "previous_65_train_15_val_mAP50_95":0.239801,
        "note":"This metric belongs to the previous development model, not FINAL80."
    }
}
(DEPLOY_DIR/"deployment_meta.json").write_text(json.dumps(meta,indent=2,ensure_ascii=False),encoding="utf-8")

taxonomy={"names":{i:n for i,n in enumerate(CLASS_NAMES)},"nc":13}
(DEPLOY_DIR/"taxonomy.yaml").write_text(yaml.safe_dump(taxonomy,sort_keys=False,allow_unicode=True),encoding="utf-8")

print(json.dumps(meta,indent=2,ensure_ascii=False))


{
  "model": "RF-DETR Medium",
  "resolution": 640,
  "num_classes": 13,
  "class_names": [
    "excavator",
    "dump_truck",
    "truck",
    "loader",
    "bulldozer",
    "motor_grader",
    "roller",
    "concrete_mixer",
    "telehandler",
    "piling_machine",
    "crane_manipulator",
    "mobile_crane",
    "tower_crane"
  ],
  "seed": 42,
  "epochs": 30,
  "lr": 2e-05,
  "lr_encoder": 1e-05,
  "use_ema": true,
  "deployment_weights": "last_ema.pth copied as final_model.pth",
  "external_checkpoint": "/kaggle/input/notebooks/maryewwe/lct-rfdetr-medium-640-finetune/rfdetr_medium_640_finetune/checkpoint_best_ema.pth",
  "external_checkpoint_sha256": "8424a45c4adc548fe4bc82bd00f4fe6a5efb171771bdb3be18c278ffc9fe4145",
  "final_model_sha256": "94a68be921b4e24cf12f362278c97bc2ae4ac47b1872a306ad9189e8e3702058",
  "development_images": 80,
  "former_train_images": 65,
  "former_val_images": 15,
  "test_images_used": 0,
  "validation_note": "No held-out validation exists for FINAL80; va

In [8]:
inference_py = 'from pathlib import Path\nfrom PIL import Image\nfrom rfdetr import RFDETRMedium\n\nMODEL_PATH = Path("final_model.pth")\nTHRESHOLD = 0.25\n\nmodel = RFDETRMedium(\n    pretrain_weights=str(MODEL_PATH),\n    resolution=640,\n    num_classes=13,\n)\n\ndef detect(image_path: str, threshold: float = THRESHOLD):\n    image = Image.open(image_path).convert("RGB")\n    det = model.predict(image, threshold=threshold)\n    rows = []\n    if len(det) == 0:\n        return rows\n    names = det.data["class_name"]\n    for box, score, raw_name in zip(det.xyxy, det.confidence, names):\n        name = raw_name.decode("utf-8") if isinstance(raw_name, bytes) else str(raw_name)\n        x1, y1, x2, y2 = map(float, box)\n        rows.append({\n            "class": name,\n            "confidence": float(score),\n            "bbox_xyxy": [x1, y1, x2, y2],\n        })\n    return rows\n\nif __name__ == "__main__":\n    import sys, json\n    if len(sys.argv) != 2:\n        raise SystemExit("Usage: python inference_example.py path/to/image.jpg")\n    print(json.dumps(detect(sys.argv[1]), indent=2, ensure_ascii=False))\n'
deploy_readme = 'LCT FINAL80 DEPLOYMENT MODEL\n\nMODEL\nRF-DETR Medium @640\nFile: final_model.pth\nWeights: final EMA weights after fixed 30-epoch FINAL80 training.\n\nTRAINING DATA\n65 former organizer TRAIN + 15 former organizer VAL = 80 development images.\n20 TEST images were not used by this notebook.\n\nMETRIC NOTE\nFINAL80 has no held-out validation set.\nDo NOT claim FINAL80 mAP50-95 = 0.2398.\n0.2398 belongs to the previous 65-train / 15-held-out-val development model.\n\nINFERENCE\nRecommended initial deployment:\n- full-frame only\n- resolution 640\n- confidence threshold 0.25\n\nExperimental crop/ROI fusion is not baked into this checkpoint.\n'

(DEPLOY_DIR/"inference_example.py").write_text(inference_py,encoding="utf-8")
(DEPLOY_DIR/"requirements.txt").write_text("rfdetr==1.10.1\npillow\n",encoding="utf-8")
(DEPLOY_DIR/"README_DEPLOY.txt").write_text(deploy_readme,encoding="utf-8")

print((DEPLOY_DIR/"README_DEPLOY.txt").read_text())


LCT FINAL80 DEPLOYMENT MODEL

MODEL
RF-DETR Medium @640
File: final_model.pth
Weights: final EMA weights after fixed 30-epoch FINAL80 training.

TRAINING DATA
65 former organizer TRAIN + 15 former organizer VAL = 80 development images.
20 TEST images were not used by this notebook.

METRIC NOTE
FINAL80 has no held-out validation set.
Do NOT claim FINAL80 mAP50-95 = 0.2398.
0.2398 belongs to the previous 65-train / 15-held-out-val development model.

INFERENCE
Recommended initial deployment:
- full-frame only
- resolution 640
- confidence threshold 0.25

Experimental crop/ROI fusion is not baked into this checkpoint.



In [9]:
# Smoke test on ONE TRAIN image only. Not an evaluation.
del model
if torch.cuda.is_available(): torch.cuda.empty_cache()

deploy_model=RFDETRMedium(
    pretrain_weights=str(FINAL_MODEL),
    resolution=640,
    num_classes=13,
)

smoke_img=train_images[0]
with Image.open(smoke_img) as im:
    pil=im.convert("RGB")

det=deploy_model.predict(pil,threshold=0.25)
print("Smoke image:",smoke_img.name)
print("Detections @0.25:",len(det))
print("FINAL MODEL LOAD/PREDICT: OK")


[2026-09-28 22:03:11] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-09-28 22:03:11] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-09-28 22:03:12] [WARNING] rf-detr - Model is not optimized for inference. Latency may be higher than expected. For full GPU throughput (e.g. ~8x on T4 via FP16 Tensor Cores), call model.inference(dtype=torch.float16).


Smoke image: Screenshot_1.png
Detections @0.25: 1
FINAL MODEL LOAD/PREDICT: OK


In [10]:
bundle=shutil.make_archive(
    str(WORK/"final80_deployment_bundle"),
    "zip",
    root_dir=DEPLOY_DIR,
)

print("\nFINAL80 DONE")
print("MODEL:",FINAL_MODEL)
print("BUNDLE:",bundle)
print("TEST USED: NO")



FINAL80 DONE
MODEL: /kaggle/working/final80_deployment/final_model.pth
BUNDLE: /kaggle/working/final80_deployment_bundle.zip
TEST USED: NO


После выполнения скачать:

`/kaggle/working/final80_deployment_bundle.zip`

Внутри:
- `final_model.pth`
- `inference_example.py`
- `requirements.txt`
- `taxonomy.yaml`
- `deployment_meta.json`
- `final80_manifest.csv`
- `README_DEPLOY.txt`

Команде лучше передавать весь ZIP, а не training notebook.
